In [4]:
import pandas as pd

In [5]:
df = pd.read_csv("data/activities_all.csv")

In [6]:
df["start_date_local"] = pd.to_datetime(df["start_date_local"], errors="coerce")  # [web:236]
data_mais_antiga = df["start_date_local"].min()  # [web:268]

print(data_mais_antiga)

2017-03-03 07:20:32+00:00


In [7]:
# --- parâmetros ---
ANO = 2025
NOME = "Diego Galdino"   # ajuste para o valor exato que aparece em df["nome"]

In [8]:
# --- prepara base ---
base = df.copy()
base["start_date_local"] = pd.to_datetime(base["start_date_local"], errors="coerce")  # [web:236]

# normaliza nome (opcional, ajuda se tiver maiúsc/minúsc diferentes)
base["nome_norm"] = base["nome"].astype(str).str.strip().str.lower()
nome_norm = NOME.strip().lower()

In [9]:
# --- filtra (ano + pessoa) ---
df_2025 = base[
    (base["start_date_local"].dt.year == ANO) &  # [web:237]
    (base["nome_norm"] == nome_norm)
].copy()  # [web:254]

# --- KPIs que você pediu ---
total_atividades = len(df_2025)

pct_por_esporte = (
    df_2025["sport_type"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

# distância (Strava: normalmente em metros -> km)
km_por_esporte = (
    df_2025.groupby("sport_type")["distance"]
    .sum()
    .div(1000)
    .sort_values(ascending=False)
)

# corrida e bike (ajuste os labels conforme seu sport_type real)
km_corrida = df_2025.loc[df_2025["sport_type"].isin(["Run"]), "distance"].sum() / 1000
km_bike = df_2025.loc[df_2025["sport_type"].isin(["Ride", "VirtualRide"]), "distance"].sum() / 1000

# tempo em atividade (moving_time em segundos -> horas)
segundos_atividade = df_2025["moving_time"].sum()
horas_atividade = segundos_atividade / 3600



In [10]:
# % do seu tempo em 2025 (sobre o ano inteiro)
segundos_ano = 365 * 24 * 60 * 60
pct_do_ano_em_atividade = (segundos_atividade / segundos_ano) * 100

print(f"Pessoa: {NOME} | Ano: {ANO}")
print(f"Total de atividades: {total_atividades}")
print(f"Tempo total em atividade: {horas_atividade:.1f} h")
print(f"% do ano em atividade: {pct_do_ano_em_atividade:.3f}%")
print(f"Distância corrida (Run): {km_corrida:.1f} km")
print(f"Distância bike (Ride/VirtualRide): {km_bike:.1f} km")

print("\n% de atividades por esporte:")
print(pct_por_esporte)

print("\nKm por esporte:")
print(km_por_esporte)

Pessoa: Diego Galdino | Ano: 2025
Total de atividades: 205
Tempo total em atividade: 226.9 h
% do ano em atividade: 2.590%
Distância corrida (Run): 962.0 km
Distância bike (Ride/VirtualRide): 1202.4 km

% de atividades por esporte:
sport_type
Run               48.3
WeightTraining    37.6
Ride              12.7
VirtualRide        1.0
Walk               0.5
Name: proportion, dtype: float64

Km por esporte:
sport_type
Ride              1165.1422
Run                962.0072
VirtualRide         37.2214
Walk                 3.5067
WeightTraining       0.0000
Name: distance, dtype: float64


In [12]:
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

OUTDIR = Path("out")
OUTDIR.mkdir(exist_ok=True)

# -----------------------
# 1) Gráfico: % por esporte (donut)
# -----------------------
fig, ax = plt.subplots(figsize=(4.2, 4.2), dpi=200)
vals = pct_por_esporte.values
labels = pct_por_esporte.index.astype(str)

wedges, _ = ax.pie(vals, startangle=90)
ax.add_artist(plt.Circle((0, 0), 0.65, color="white"))
ax.set_title("% de atividades por esporte")
ax.legend(wedges, [f"{l} ({v:.1f}%)" for l, v in zip(labels, vals)],
          loc="center left", bbox_to_anchor=(1.05, 0.5), frameon=False)
plt.tight_layout()
pie_path = OUTDIR / "pct_esporte.png"
fig.savefig(pie_path, dpi=200, bbox_inches="tight")  # export estático [web:36]
plt.close(fig)

# -----------------------
# 2) Gráfico: km por esporte (barras horizontais)
# -----------------------
fig, ax = plt.subplots(figsize=(5.2, 3.6), dpi=200)
km_por_esporte.sort_values().plot(kind="barh", ax=ax)
ax.set_title("Km por esporte")
ax.set_xlabel("Km")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
bar_path = OUTDIR / "km_esporte.png"
fig.savefig(bar_path, dpi=200, bbox_inches="tight")  # bbox_inches para cortar “sobras” [web:36]
plt.close(fig)

# -----------------------
# 3) Montagem do infográfico (canvas)
# -----------------------
W, H = 1400, 900
bg = Image.new("RGB", (W, H), "white")
draw = ImageDraw.Draw(bg)

# Fonte: tente uma TTF do sistema; se não tiver, cai no default
try:
    font_title = ImageFont.truetype("arial.ttf", 52)
    font_sub = ImageFont.truetype("arial.ttf", 28)
    font_kpi = ImageFont.truetype("arial.ttf", 34)
    font_small = ImageFont.truetype("arial.ttf", 22)
except:
    font_title = ImageFont.load_default()
    font_sub = ImageFont.load_default()
    font_kpi = ImageFont.load_default()
    font_small = ImageFont.load_default()

# Header
draw.text((60, 40), f"Retrospectiva {ANO}", fill="black", font=font_title)
draw.text((60, 105), f"{NOME} • Strava", fill="black", font=font_sub)

# KPI cards (texto)
kpi_text = (
    f"Total de atividades: {total_atividades}\n"
    f"Tempo total: {horas_atividade:.1f} h\n"
    f"% do ano em atividade: {pct_do_ano_em_atividade:.3f}%\n"
    f"Corrida: {km_corrida:.1f} km\n"
    f"Bike: {km_bike:.1f} km"
)
draw.multiline_text((60, 170), kpi_text, fill="black", font=font_kpi, spacing=10)  # multiline_text [web:40][web:46]

# Inserir gráficos
pie_img = Image.open(pie_path).convert("RGB")
bar_img = Image.open(bar_path).convert("RGB")

# Redimensiona para caber bem (mantendo proporção aproximada)
pie_img = pie_img.resize((520, 520))
bar_img = bar_img.resize((720, 420))

bg.paste(pie_img, (820, 60))
bg.paste(bar_img, (620, 470))

# Rodapé
draw.text((60, H - 50), "Gerado com Python (pandas + matplotlib + pillow)", fill="gray", font=font_small)

out_path = OUTDIR / f"infografico_{ANO}.png"
bg.save(out_path)
print("Salvo em:", out_path)


Salvo em: out\infografico_2025.png


In [6]:
df["nome"] = df["nome"].str.lower()

In [7]:
df_diego = df[df["nome"].str.contains("diego", case=False, na=False)].copy()


In [9]:
df_diego.info()

<class 'pandas.core.frame.DataFrame'>
Index: 100 entries, 0 to 99
Data columns (total 62 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   resource_state                 100 non-null    int64  
 1   name                           100 non-null    object 
 2   distance                       100 non-null    float64
 3   moving_time                    100 non-null    int64  
 4   elapsed_time                   100 non-null    int64  
 5   total_elevation_gain           100 non-null    float64
 6   type                           100 non-null    object 
 7   sport_type                     100 non-null    object 
 8   workout_type                   61 non-null     float64
 9   device_name                    98 non-null     object 
 10  id                             100 non-null    int64  
 11  start_date                     100 non-null    object 
 12  start_date_local               100 non-null    object 
 

In [ ]:
df_diego = df_diego.drop(columns=["coluna1", "coluna2"])
